TLS sta per **Transport Layer Security**, il suo obiettivo è proteggere la comunicazione tra due applicazioni.  
Il collegamento con RADIUS sta nel fatto che, come anticipato, conviene affidare la protezione della comunicazione ad un protocollo dedicato, e TLS si occupa proprio di questo.

Ci poniamo due obiettivi ai fini del corso:
- Capire il funzionamento di un protocollo di sicurezza ampiamente utilizzato come TLS
- imparare a progettare protocolli sicuri, studiando sia le scelte corrette che gli errori

In questo senso ci concentreremo sulle versioni storiche di TLS e sulle loro vulnerabilità.

**Da SSL a TLS**:  
SSL sta per **Secure Sockets Layer**. Nasce in Netscape negli anni '90, quando la diffusione del Web rese importante proteggere le comunicazioni e le transazioni online.

Si parte da SSL 1.0 del '94, mai distribuito pubblicamente, SSL 2.0 del '95, distribuito ma con gravi vulnerabilità, e SSL 3.0 del '96 rappresentò una revisione sostanziale del progetto.  
TLS 1.0 arrivl nel 1999 come standard IETF e sviluppato partendo da SSL 3.0. Nel 2006 con TLS 1.1 si introdussero alcune migliorie, e nel 2008 con TLS 1.2 si introdusse maggiore flessibilità crittografica e ulteriori miglioramenti. Fu con TLS 1.3 del 2018 che si arrivò ad una revisione importante dell'handshake e della protezione dati.

TLS 1.3 cambia molto più di quanto non suggerisca il numero di versione: tra le scelte importanti l'utilizzo esclusivo di **AEAD (Authenticated Encryption with Associated Data)** per proteggere i record, come vedremo.

Un altro protocollo importante è **DTLS (Datagram Transport Layer Security)**, che è una variante di TLS per i protocolli a datagramma come UDP. Infatti TLS classico assume un trasporto affidabile e ordinato come TCP; UDP invece sappiamo trasportare datagrammi che possono andare persi o arrivare in disordine --> DTLS adatta il progetto TLS a questo tipo di trasporto.

**Dove si colloca TLS**? Nell'esempio classico del Web, la pila sarebbe:
```
HTTP / TLS / TCP / IP
```
quindi si colloca tra livello applicativo e di trasporto.  
Qui è da chiarire quindi che **TLS non rende sicuro l'intero protocollo TCP**: protegge solo i dati che gli vengono affidati. Gli header TCP/IP e altri metadati della comunicazione restano esterni alla protezione del contenuto applicativo. 

Inoltre TLS non è limitato al Web: può essere utilizzato per proteggere qualsiasi protocollo applicativo, es. in metodi di autenticazione come EAP-TLS, anche prima che l'utente disponga di un normale accesso IP.

Il prof. critica una scelta storica: **l'associazione di una porta distinta alla versione protetta di un servizio**.  
L'esempio preso in causa è quello di HTTP e HTTPS: il primo è sulla porta 80, il secondo sulla 443. HTTPS mantiene HTTP come protocollo applicativo, ne protegge solo lo scambio tramite TLS. Quindi HTTPS non sostituisce tutte le operazioni HTTP con operazioni nuove (es. richieste come GET e POST restano HTTP).  

Questo approccio è stato ripetuto anche per altri servizi. Tuttavia secondo il prof l'integrazione sarebbe dovuta essere più simile a un meccanismo che identificasse prima il livello di sicurezza, e solo poi il protocollo applicativo trasportato.  

L'alternativa migliore secondo il prof si vede nel comportamento di IPsec.  
Sappiamo come normalmente un pacchetto IP contenga un header IP e un payload, che può essere un datagramma UDP o un segmento TCP, in particolare nell'header IP c'è il campo Protocol che indica il tipo di payload trasportato.

Con IPsec si introduce un livello di sicurezza tra IP e il trasporto, quindi il payload del pacchetto IP non è più un datagramma UDP o un segmento TCP, ma un pacchetto IPsec che viene indicato quindi come tale nell'header, ad es:
```
IP / AH / TCP / HTTP
```
dove AH è il protocollo di autenticazione e integrità IPsec.  
Quindi ogni livello indica esplicitamente quale protocollo segue.  
Con HTTPS invece avevamo:
```
IP / TCP / TLS / HTTP
```
per cui IP indica direttamente TCP, ma non esiste una porta TCP unica per ogni comunicazione TLS proprio a causa della scelta di distinguere i servizi protetti da TLS con una porta dedicata.

## Gli obiettivi di TLS
TLS rappresenta in un certo senso "due protocolli in uno":
1. **Handshake protocol**: prima di trasferire i dati protetti, client e server devono accordarsi in termini di:
   - parametri crittografici da utilizzare
   - stabilire materiale segreto condiviso
   - effettuare l'autenticazione prevista
  
  finora abbiamo assunto infatti che i partecipanti disponessero già di una chiave condivisa, ma nel Web questa assunzione non è realistica in quanto un browser può contattare server mai incontrati prima.  
  Si parlerà in particolare di crittografia asimmetrica e dei certificati.  
  **Autenticare chi?** Nell'uso del Web abituale, è il client a verificare l'identità del server. L'autenticazione TLS del client può esser prevista ma non è obbligatoria ad ogni utilizzo.  
  **Perché negoziare sugli algoritmi?** TLS non coincide con un unico cifrario specifico: permette di scegliere tra combinazioni di algoritmi supportate dalla versione e dalle configurazioni dei partecipanti.  
  Vedremo come potenzialmente un attaccante può alterare la negoziazione per far scegliere una protezione più debole --> la negoziazione a sua volta deve essere protetta.
2. **Record protocol**: una volta stabilite chiavi e parametri, **TLS protegge il traffico usando principalmente crittografia simmetrica**.  
Il motivo principale è l'efficienza: la crittografia simmetrica è adatta a grandi quantità di dati.  

Quindi TLS offre come proprietà fondamentali **riservatezza, integrità e autenticazione dei dati**. Nel modello storico analizzato queste proprietà sono ottenute tramite cifratura e HMAC separati, mentre con TLS 1.3 si utilizza un unico meccanismo AEAD che fornisce tutte e tre le proprietà in un'unica operazione.

### TLS protocol stack
La struttura interna TLS presenta il **Record Protocol** come il livello comune, che trasporta diversi tipi di contenuto:
- **Handshake Protocol** per negoziare come detto parametri, stabilire segreti ed effettuare l'autenticazione
- **Alert Protocol** per notificare errori o condizioni particolari, inclusa la chiusura della connessione
- **Change Cipher Spec**: segnala, nel modello storico, il passaggio allo stato crittografico controllato
- **Application Data**: trasporta i dati applicativi protetti, es. richieste e risposte HTTP.

Quindi il Record Protocol non viene usato solo dopo l'handshake, ma trasporta anche i messaggi dell'handshake stesso. 

<img src="imgs/TLS_stack.png" width="400">

Notare quindi come **i primi messaggi non sono tutti cifrati**, in quanto inizialmente le chiavi non sono ancora state stabilite.

**Attenzione al fatto che il modello di TLS che vediamo riguarda TLS fino alla versione 1.2: TLS 1.3 differisce.**

Le **operazioni storiche del Record Protocol** (assumendo che l'handshake sia già stato completato) sono le seguenti:
1. ricevere i dati applicativi
2. frammentarli ed eventualmente comprimerli
3. aggiungere un MAC, per poi cifrare sia i dati che il MAC
4. aggiungere l'header del record e affidare il risultato al trasporto

**Frammentazione**: i dati vengono suddivisi in porzioni gestibili. Questa è una frammentazione al livello TLS e non coincide quindi con la segmentazione TCP.

**Compressione**: l'idea storica è ridurre la dimensione dei dati prima la cifratura (prima della cifratura perché un buon ciphertext non presenta regolarità e quindi non può essere compresso nel senso classico). Vedremo però come questo passaggio abbia portato ad una vulnerabilità (CRIME) e quindi sia stato eliminato in TLS 1.3.

**Aggiunta del MAC e cifratura**: sia $m$ il frammento dopo l'eventuale compressione. In forma semplificata:
$$t = MAC_{K_M}(m) \quad \text{e} \quad c = E_{K_E}(m || t)$$
ossia prima si calcola prima il MAC del frammento, e poi si cifra il frammento insieme al MAC ($t$ è il tag di autenticazione, si produce usando la chiave segreta usando la logica del MAC --> permetterò al destinatario di verificare l'integrità e l'autenticità del frammento; mentre $c$ è il ciphertext che viene trasmesso).  
Se si usa un cifrario a blocchi, nella costruzione concreta può esser necessario aggiungere padding prima di cifrare.  
Infine viene aggiunto l'header del record:
$$\text{record} = \text{header} || c$$

**Due grandi errori in questo meccanismo**:
- comprimere i dati prima di cifrarli può aprire un canale laterale (i.e. fonte indiretta di informazioni su un segreto)
- la combinazione MAC-then-encrypt può creare difficoltà di sicurezza, in quanto il MAC non è visibile al destinatario prima della decifratura.

Vedremo questi attacchi in modo approfondito successivamente, per ora focalizziamoci sui problemi

#### Primo problema: la compressione può rivelare informazioni attraverso la lunghezza
In questo senso l'esempio classico è l'attacco CRIME.  
Sappiamo come la cifratura nasconda il contenuto, ma *non necessariamente la lunghezza del contenuto cifrato. La compressione invece produce lunghezze che dipendono dalla regolarità e ripetizioni dei dati.

Supponiamo che dati segreti, come un cookie, vengano compressi insieme a dati influenzabili dall'attaccante.  
Allora l'attaccante può provare diverse stringhe e osservare la lunghezza del risultato risultato cifrato. Se tra le stringhe provate una riproduce una parte del segreto --> il compressore sfrutta la ripetizione e produce un output più corto --> la variazione della lunghezza diventa indizio sul contenuto nascosto. 

Quindi CRIME sfrutta un canale laterale legato alla compressione.  
Ciò **non significa automaticamente che la variazione della lunghezza sia sufficiente a rivelare il contenuto segreto**: l'attacco richiede infatti condizioni appropriate, tra cui la possibilità di influenzare i dati compressi insieme al segreto e osservare il risultato.

#### Secondo problema: cifrare e autenticare non sono operazioni intercambiabili
Si confrontano in questo senso tre costruzioni. Siano $m = \text{message}$, $K_E = \text{encryption key}$, $K_M = \text{MAC key}$.  
Allora:
1. **MAC-then-encrypt** calcola il MAC del messaggio e poi cifra entrambi:
   $$t = MAC_{K_M}(m) \quad \text{e} \quad c = E_{K_E}(m || t)$$
2. **Encrypt-then-MAC** cifra il messaggio e poi autentica il ciphertext:
   $$(c, MAC_{K_M}(c))$$
3. **Encrypt-and-MAC** cifra il plaintext e calcola il MAC separatamente:
   $$(E_{K_E}(m), MAC_{K_M}(m))$$

Questi schemi sono associati a costruzioni storiche di TLS, IPsec e SSH. 

**MAC-then-encrypt**:  
Il destinatario deve decifrare prima di verificare il MAC interno:
$$t = MAC_{K_M}(m) \quad \text{e} \quad c = E_{K_E}(m || t)$$
In costruzioni con padding, questo può comportare **elaborazione di dati manipolati** prima che la loro autenticità sia verificata. Vedremo degli attacchi concreti nelle lezioni successive, il punto da capire è che il fatto che MAC e cifratura siano operazioni sicure prese separatamente non implica che la loro composizione sia sicura.

**Encrypt-then-MAC**:  
$c = E_{K_E}(m)$ e poi $t = MAC_{K_M}(c)$, quindi il destinatario può verificare il tag prima di decifrare e scartare ciphertext alterati.  
Questa rappresenta una composizione generale robusta. 

**Encrypt-and-MAC**:
$$ c = E_{K_E}(m) \quad \text{e} \quad t = MAC_{K_M}(m)$$
vengono quindi trasmessi $(c, t)$. Il tag esterno riguarda quindi direttamente il plaintext e può pertanto rivelare informazioni sul messaggio anche se il ciphertext è sicuro.

**Esercizio: mostrare il problema di Encrypt-and-MAC con IND-CPA**  
Consideriamo il modello semplificato:
$$Enc'(m) = (E_{K_E}(m), MAC_{K_M}(m))$$
con MAC deterministico.  
L'attaccante allora può scegliere due messaggi distinti di uguale lunghezza $m_0, m_1$ e chiedere, prima della challenge, di cifrarli registrando i tag:
$$t_0 = MAC_{K_M}(m_0) \quad t_1 = MAC_{K_M}(m_1)$$
Si scelgono in particolare due messaggi che producono tag distinti.

Il challenger allora cifra poi uno dei due messaggi, scelto a caso mediante un bit segreto $b$:
$$(c^*, t^*) = (E_{K_E}(m_b), MAC_{K_M}(m_b))$$
L'attaccante allora confronta il tag: se $t^* = t_0$ allora $b=0$, altrimenti $b=1$ --> distingue i due messaggi senza decifrare e senza conoscere la chiave del MAC.

Ciò non viola l'autenticità del MAC, in quanto l'attaccante non ne sta falsificando uno, ma mostra come stia usando tag legittimi per riconoscere il plaintext. L'esercizio cioè mostra come un MAC sicuro contro la falsificazione non garantisce che il suo tag nasconda il messaggio, almeno nel senso di IND-CPA.

(lo stesso discorso non si può fare ad es. con Encrypt-then-MAC in quanto anche se l'attaccante chiedesse all'oracolo di dargli il ciphertext di un messaggio $m_1$, la cifratura di uno stesso messaggio produce un ciphertext diverso ogni volta, quindi non può distinguere i due messaggi)

#### Struttura del Record e del tag
Abbiamo prima visto cifratura e compressione con alcune semplificazioni. Il vero meccanismo prevede che il frammento (eventualmente compresso) venga autenticato tramite HMAC e poi cifrato insieme al tag:
$$t = HMAC_{K_M}(\text{header} || m) \quad c = E_{K_C}(m || t || \text{padding})$$
dove il padding serve quando richiesto dalla modalità utilizzata (tipo CBC).

Le scelte progettuali valide in questo senso sono **separazione delle chiavi per autenticazione e cifratura** $K_M \neq K_C$ e **possibilità di cambiare gli algoritmi**.

Il record trasmesso ha un header di 5 byte, contenente:
- **Content Type**: indica il tipo di contenuto trasportato (handshake, alert, application data, ecc.)
- **Version**: indica la versione TLS in uso
- **Length**: indica la lunghezza del contenuto del record

Un altro obiettivo del MAC è **proteggere anche dalla ripetizione dei record**. Infatti un tag valido dimostra che il messaggio non è stato modificato senza autorizzazione, ma non dimostra da solo che sia nuovo. 

Es. messaggio "paga 1000 euro" --> un attaccante potrebbe tentare di ripetere il messaggio.  

Per distringuere tra diverse occorrenze, TLS usa **numeri di sequenza**: ogni direzione della comunicazione ha un proprio contatore $s_{C->S}$ e $s_{S->C}$ per cui il mittente tiene il contatore dei record inviati, mentre il destinatario il rispettivo di quelli ricevuti.

Si tratta di contatori a 64 bit, inizializzati a zero con un certo stato crittografico, che non devono ovviamente ricominciare da zero con le stesse condizioni crittografiche. Per il calcolo del MAC viene incluso questo valore, più precisamente:
$$t = HMAC_{K_M}(s || type || version || length || m)$$
in tal modo se un record era stato autenticato con $s=5$ --> ripresentarlo quando il destinatario si aspetta $s=9$ fa si che non venga prodotto un MAC valido per via di quella differenza.